In [0]:
%run ./01-config

In [0]:
class Bronze():
    def __init__(self, env, db_name):
        Conf = Config()
        self.landing_zone = Conf.base_dir_data + "/raw"
        self.checkpoint_base = Conf.base_dir_data + "/checkpoints"
        self.catalog = env
        self.db_name = db_name
        spark.sql(f"USE {self.catalog}.{self.db_name}")

    def consume_crm_cust_info(self):
        from pyspark.sql import functions as F
        schema = "cst_id INT, cst_key string, cst_firstname string, cst_lastname string, cst_marital_status string, cst_gndr string, cst_create_date DATE"

        df = (spark.read
                .format("csv")
                .schema(schema)
                .option("header", "true")
                .load(self.landing_zone + "/cust_info")
        )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.crm_cust_info"))
        
    def consume_crm_prd_info(self):
        from pyspark.sql import functions as F
        schema = "prd_id INT, prd_key string, prd_name string, prd_cost INT, prd_line string, prd_start_dt DATE, prd_end_dt DATE"

        df = (spark.read
                .format("csv")
                .schema(schema)
                .option("header", "true")
                .load(self.landing_zone + "/prd_info")
        )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.crm_prd_info"))
        
    def consume_crm_sales_details(self):
        from pyspark.sql import functions as F
        schema = "sls_ord_num string, sls_prd_key	string, sls_cust_id	INT, sls_order_dt INT, sls_ship_dt	INT, sls_due_dt	INT, sls_sales INT, sls_quantity INT, sls_price INT"

        df = (spark.read
                .format("csv")
                .schema(schema)
                .option("header", "true")
                .load(self.landing_zone + "/sales_details")
        )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.crm_sales_details"))
        
    def consume_erp_cust_az12(self):
        from pyspark.sql import functions as F
        schema = "cid string, bdate DATE, gen string"

        df = (spark.read
                .format("csv")
                .schema(schema)
                .option("header", "true")
                .load(self.landing_zone + "/CUST_AZ12")
        )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.erp_cust_az12"))
        
    def consume_erp_loc_a101(self):
        from pyspark.sql import functions as F
        schema = "cid string, cntry string"

        df = (spark.read
                .format("csv")
                .schema(schema)
                .option("header", "true")
                .load(self.landing_zone + "/LOC_A101")
        )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.erp_loc_a101"))
        
    def consume_erp_px_cat_glv2(self):
        from pyspark.sql import functions as F
        schema = "id string, cat string, subcat string, maintenance string"

        df = (spark.read
                .format("csv")
                .schema(schema)
                .option("header", "true")
                .load(self.landing_zone + "/PX_CAT_G1V2")
        )

        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{self.catalog}.{self.db_name}.erp_px_cat_glv2"))
        
    def consume(self):
        import time
        start = int(time.time())
        print(f"\nStarting bronze layer consumption...")
        self.consume_crm_cust_info()
        self.consume_crm_prd_info()
        self.consume_crm_sales_details()
        self.consume_erp_cust_az12()
        self.consume_erp_loc_a101()
        self.consume_erp_px_cat_glv2()
        end = int(time.time())
        print(f"\nCompleted bronze layer consumption {end - start} seconds")

    def assert_count(self, table_name, expected_count, filter = True):
        print(f"\nValidating record counts in {table_name}...", end = '')
        actual_count = spark.read.table(f"{self.catalog}.{self.db_name}.{table_name}").where(filter).count()
        assert actual_count == expected_count, f"Expected {expected_count:, } records, found {actual_count:,} in {table_name} where {filter}"
        print(f"Found {actual_count:,} / {expected_count:,} records where {filter}: Success")

    def validate(self):
        import time
        start = int(time.time())
        print(f"\nValidating bronze layer records...")
        checks = {
        "crm_cust_info":     "cust_info/cust_info.csv",
        "crm_prd_info":      "prd_info/prd_info.csv",
        "crm_sales_details": "sales_details/sales_details.csv",
        "erp_cust_az12":     "CUST_AZ12/CUST_AZ12.csv",
        "erp_loc_a101":      "LOC_A101/LOC_A101.csv",
        "erp_px_cat_g1v2":   "PX_CAT_G1V2/PX_CAT_G1V2.csv"
        }

        for table, path in checks.items():
            src_count = (spark.read.format("csv")
                            .option("header", "true")
                            .load(f"{self.landing_zone}/{path}")
                            .count())
            self.assert_count(table, src_count)
        end = int(time.time())
        print(f"\nBronze layer records validation completed in {end - start} seconds")
